## Init session

In [ ]:
%load_ext autoreload
%autoreload 2

### Install dependencies

In [ ]:
!chmod +x install.sh
! ./install.sh > /dev/null 2>&1

### Import packages

In [ ]:
import os
import boto3
import subprocess

from pathlib import Path
from random import randint

from rich.pretty import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import torch

print(torch.version.cuda)           
print(torch.backends.cudnn.version()) 
print(torch.cuda.is_available())  

from sklearn.model_selection import train_test_split

import model as ic

## Download data from S3 bucket

In [ ]:
# Bucket ids
mybucket = "maximelenormand"
key='PCIKAKXXF3CIF2B0ECT5'
secret='NVg09XXnVjdSP87hAA3o6Vh3oGTt4eSxOO8TpUtW'
token='eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJhY2Nlc3NLZXkiOiJQQ0lLQUtYWEYzQ0lGMkIwRUNUNSIsImFsbG93ZWQtb3JpZ2lucyI6WyIqIl0sImF1ZCI6WyJtaW5pby1kYXRhbm9kZSIsIm9ueXhpYSIsImFjY291bnQiXSwiYXV0aF90aW1lIjoxNzc1MTE2NTMwLCJhenAiOiJvbnl4aWEiLCJjbmYiOnsiamt0IjoieF8zaGM4MDNkNVJGMDJJQnF3SjNKLU1zT0ZFVjRUTjZObVpsYWl1SHBXUSJ9LCJlbWFpbCI6Im1heGltZS5sZW5vcm1hbmRAaW5yYWUuZnIiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZXhwIjoxNzc1NzI1NzMwLCJmYW1pbHlfbmFtZSI6Ikxlbm9ybWFuZCIsImdpdmVuX25hbWUiOiJNYXhpbWUiLCJncm91cHMiOlsiVVNFUl9PTllYSUEiXSwiaWF0IjoxNzc1MTIwOTMwLCJpc3MiOiJodHRwczovL2F1dGgubGFiLnNzcGNsb3VkLmZyL2F1dGgvcmVhbG1zL3NzcGNsb3VkIiwianRpIjoib25ydHJ0OjUyMGQzMzM3LTNjNzctZTdiOC0yOGNiLTY4NzBiMWJlNTZhYyIsIm5hbWUiOiJNYXhpbWUgTGVub3JtYW5kIiwicG9saWN5Ijoic3Rzb25seSIsInByZWZlcnJlZF91c2VybmFtZSI6Im1heGltZWxlbm9ybWFuZCIsInJlYWxtX2FjY2VzcyI6eyJyb2xlcyI6WyJvZmZsaW5lX2FjY2VzcyIsInVtYV9hdXRob3JpemF0aW9uIiwiZGVmYXVsdC1yb2xlcy1zc3BjbG91ZCJdfSwicmVzb3VyY2VfYWNjZXNzIjp7ImFjY291bnQiOnsicm9sZXMiOlsibWFuYWdlLWFjY291bnQiLCJtYW5hZ2UtYWNjb3VudC1saW5rcyIsInZpZXctcHJvZmlsZSJdfX0sInJvbGVzIjpbIm9mZmxpbmVfYWNjZXNzIiwidW1hX2F1dGhvcml6YXRpb24iLCJkZWZhdWx0LXJvbGVzLXNzcGNsb3VkIl0sInNjb3BlIjoib3BlbmlkIHByb2ZpbGUgZ3JvdXBzIGVtYWlsIiwic2lkIjoiYTRmMTBhOTEtNjliNC1hYzJhLTNmNDItMGQyN2UzMmM3ZWIyIiwic3ViIjoiM2M5MDk3MWMtNDEwMC00NmNhLWI3OTYtMTg5MGU0N2NhYWZkIiwidHlwIjoiRFBvUCJ9.B3uuYsHue_lrUKYXuKT3orMv6HAHn2yRembgOe_mpFkl7JNzlkkiFXdY6ElCioqsMbhC3Fjiq6IMcCLvkMgSxQ'

s3 = boto3.client("s3",endpoint_url = 'https://minio.lab.sspcloud.fr',
                  aws_access_key_id = key, 
                  aws_secret_access_key = secret, 
                  aws_session_token = token)

# Check connection
listobjbucket = s3.list_objects_v2(Bucket=mybucket, MaxKeys=5)
if 'Contents' in listobjbucket:
    print("Connection OK!\nFiles:")
    for obj in listobjbucket['Contents']:
        print("-", obj['Key'])

# Download data.zip if needed
if not os.path.exists("/home/onyxia/work/data.zip"):
    s3.download_file(mybucket, "data.zip", "data.zip")

# Unzip if needed
if not os.path.exists("/home/onyxia/work/data"):
    subprocess.run(["unzip", "data.zip"],
                   stdout=subprocess.DEVNULL,
                   stderr=subprocess.DEVNULL)
    print("Done!")


## Load & clean data

### Import annotations

In [ ]:
tab_raw = pd.read_csv(Path(".").joinpath("data").joinpath("annotations.csv"))

print(type(tab_raw))
print(tab_raw)

tab_raw.iloc[:, 2:len(tab_raw)].sum()

### Select categories

In [ ]:
binary_columns_todrop = ["animal", "no", "water", "zoom"]
binary_columns = ["human", "anthropic", "vegetation", "rock", "snow"]
tab_raw = pd.read_csv(Path(".").joinpath("data").joinpath("annotations.csv")).drop(
    binary_columns_todrop, axis = 1
)

print(type(tab_raw))
print(tab_raw)

In [ ]:
tab_raw.iloc[:, -5:].sum()

### Select sites

In [ ]:
sites_to_keep = ["Carpathians", "French_Alps", "Stubai_Valley", "Vinschgau"]
sites_for_challenge = [""]

tab_raw_challenge = tab_raw[tab_raw["site"].isin(sites_for_challenge)].copy()
tab_raw = tab_raw[tab_raw["site"].isin(sites_to_keep)]

print(tab_raw.shape)
print(tab_raw_challenge.shape)

### Rebalance categories

In [ ]:
n = 2000
rng = np.random.default_rng(42)           

In [ ]:
# ------------------------------------------------------------------ #
# Compute weights from imbalance in the original dataframe
# ------------------------------------------------------------------ #
# p = proportion of 1s; distance from 0.5 ranges from 0 (perfect balance)
# to 0.5 (all 0s or all 1s). We map it to a weight >= 1.
# weight = 1 + k * (|p - 0.5| / 0.5)  with k controlling the max weight.
k = 4  # max additional weight on top of the baseline 1
weights = {}
# print("\nAuto-computed column weights:")
for col in binary_columns:
    p = tab_raw[col].mean()
    imbalance = abs(p - 0.5) / 0.5  # 0 = perfectly balanced, 1 = fully skewed
    weights[col] = 1 + k * imbalance
    # print(f"  {col}: proportion of 1s = {p:.3f}, weight = {weights[col]:.2f}")

In [ ]:
# ------------------------------------------------------------------ #
# Greedy balanced selection
# ------------------------------------------------------------------ #
seed = 42  
rng = np.random.default_rng(seed)
df_shuffled = tab_raw.sample(frac=1, random_state=seed).reset_index(drop=True)

#df_shuffled = tab_raw.sample(frac = 1, random_state = int(rng.integers(1e6))).reset_index(drop = True)

selected_indices = []
counts = {col: {0: 0, 1: 0} for col in binary_columns}
target = n // 2

base_tolerance = 500   
ramp = 1000

values = df_shuffled[binary_columns].values

for idx, row_vals in enumerate(values):
    if len(selected_indices) >= n:
        break

    score = 0

    for j, col in enumerate(binary_columns):
        val = int(row_vals[j])
        w = weights[col]

        score += w * (
            (target - counts[col][val])
            - (target - counts[col][1 - val])
        )

    # -------- trade-off control --------
    progress = len(selected_indices) / n
    threshold = -(base_tolerance + progress * ramp)

    if score >= threshold:
        selected_indices.append(idx)

        for j, col in enumerate(binary_columns):
            counts[col][int(row_vals[j])] += 1

#for _, row in df_shuffled.iterrows():
#    if len(selected_indices) >= n:
#        break
#
#    score = 0
#    for col in binary_columns:
#        w = weights[col]
#        val = int(row[col])
#        current = counts[col][val]
#        current_opposite = counts[col][1 - val]
#
#        if current < target_per_class:
#            score += 1 * w
#        elif current >= target_per_class and current_opposite < target_per_class:
#            score -= 1 * w
#
#    if score >= 0:
#        selected_indices.append(row.name)
#        for col in binary_columns:
#            counts[col][int(row[col])] += 1

tab = df_shuffled.loc[selected_indices].reset_index(drop = True)

selected_set = set(selected_indices)
tab_not_selected = df_shuffled.loc[~df_shuffled.index.isin(selected_set)].reset_index(drop=True)
tab_challenge = pd.concat([tab_raw_challenge, tab_not_selected],ignore_index=True)

pd.DataFrame(
    data={
        "Category": [col for col in binary_columns],
        "%": [tab[col].mean() * 100 for col in binary_columns],
        "Number": [sum(tab[col]) for col in binary_columns],
        "Total": len(tab),
    }
).sort_values("%", ascending = False)

In [ ]:
print(tab_raw.shape)
print(tab.shape)
print(tab_challenge.shape)

### Check labels

In [ ]:
pprint(tab.columns[2:])
pprint(tab_challenge.columns[2:])

## Test dataset

In [ ]:
path_to_images = Path(".").joinpath("data").joinpath("images")
path_to_images.is_dir()

In [ ]:
dataset = ic.FldDataset(data = tab, train_mode = True, test_mode = True)

In [ ]:
#pprint(dataset.transform)
rnd_data = dataset[randint(0, len(dataset) - 1)]

print(tab.iloc[:, -5:].sum())
pprint(rnd_data["labels"])
plt.imshow(rnd_data["image"])

In [ ]:
#plt.imshow(dataset[100]["image"])

## Train

### Split dataset

In [ ]:
tab_strat = tab.copy()
tab_strat["strat"] = ""
for col in tab.columns[2:]:
    #print(col)
    tab_strat["strat"] += tab_strat[col].astype(str)

#pprint(tab_strat.strat.value_counts())

In [ ]:
tab_strat_balanced = tab_strat.copy()
tab_strat_count = pd.DataFrame(tab_strat.strat.value_counts()).reset_index()
strat_tokeep = tab_strat_count[tab_strat_count['count'] >= 10]
tab_strat_balanced = tab_strat_balanced[tab_strat_balanced['strat'].isin(strat_tokeep['strat'])]

#pprint(tab_strat_balanced.strat.value_counts())

print(tab_raw.shape)
print(tab.shape)
print(tab_strat.shape)
print(tab_strat_balanced.shape)


In [ ]:
trainval, test = train_test_split(tab_strat_balanced, test_size = 0.15, random_state = 42, stratify = tab_strat_balanced["strat"])
#train, val = train_test_split(trainval, test_size = 0.18, random_state = 42, stratify = trainval["strat"])

#train = train.drop("strat", axis=1)
#val = val.drop("strat", axis=1)
test = test.drop("strat", axis=1)

print(test.shape)

### Mini-grid search

In [ ]:
minigrid = False
if minigrid:
    
    backbones = ["hf_swt_t", "hf_resnet", "hf_cnx2_t", "hf_vit_g16"]
    repetitions = [0, 1] 
    learning_rates = [1e-5, 1e-4]
    batch_sizes = [16, 32]
    
    for rep in repetitions:
        train, val = train_test_split(
            trainval,
            test_size=0.18,
            stratify=trainval["strat"],
            random_state=rep
        )
        train = train.drop("strat", axis=1)
        val = val.drop("strat", axis=1)
        
        for backbone in backbones:
            for lr in learning_rates:
                for bs in batch_sizes:
                    
                    ic.train_model(
                        train_data=train,
                        val_data=val,
                        batch_size=bs,
                        max_epochs=20,  
                        image_size=224,
                        run_owner="maxime",
                        exp_name="image_labeller",
                        backbone=backbone,
                        loss_name="bce",
                        loss_params={"alpha":0.25, "gamma":2},  
                        device=ic.get_device(),
                        checkpoints_n_saved=1,
                        learning_rate=lr,
                        early_stoper_patience=5,
                        early_stoper_min_delta=0.001,
                        use_lr_finder=False,
                        lr_scheduler_step=10,
                        lr_scheduler_gamma=0.85,
                        print_steps="print",
                        log_progress=False,
                        plot_loss=False,
                        num_workers=10,
                    )


### Load experiment data

In [ ]:
runs = (
    mlflow.search_runs(
        search_all_experiments = True,
        experiment_names = ["image_labeller"],
        order_by = [f"params.F1_weighted_avg DESC"],
    )
    .assign( # Cast metrics to float numbers
        **{
            k: (lambda x, col=k: x[col].astype(np.float32))
            for k in [
                "params.F1_weighted_avg",
                "params.F1_micro_avg",
                "params.F1_macro_avg",
                "params.F1_samples_avg",
                "params.F1_snow",
                "params.F1_vegetation",
                "params.F1_anthropic",
                "params.F1_human",
                "params.F1_rock",
            ]
        }
    )
)

runs

### Display mean and standard deviation for each backbone

In [ ]:
runs.groupby(["params.backbone"]).agg(
    {
        k: ["mean", "std"]
        for k in [
            "params.F1_weighted_avg",
            "params.F1_micro_avg",
            "params.F1_macro_avg",
            "params.F1_samples_avg",
            "params.F1_snow",
            "params.F1_vegetation",
            "params.F1_anthropic",
            "params.F1_human",
            "params.F1_rock",
        ]
    }
).reset_index()

### Train 50 models for hf_swt_t

In [ ]:
trainswt = False
if trainswt:
    
    for rep in [11,12,13,14,15]:
        train, val = train_test_split(
            trainval,
            test_size=0.18,
            stratify=trainval["strat"],
            random_state=rep
        )
        train = train.drop("strat", axis=1)
        val = val.drop("strat", axis=1)
        
        for _ in list(range(10)):
            ic.train_model(
                train_data = train,
                val_data = val,
                batch_size = 32,
                max_epochs = 100,
                image_size = 224,
                run_owner = "maxime",
                exp_name = "trainswt",
                backbone = "hf_swt_t",
                loss_name = "bce",
                loss_params = {"alpha": 0.25, "gamma": 2},
                device = ic.get_device(),
                checkpoints_n_saved = 1,
                learning_rate = 0.00001,
                early_stoper_patience = 10,
                early_stoper_min_delta = 0.001,
                use_lr_finder = False,
                lr_scheduler_step = 10,
                lr_scheduler_gamma = 0.85,
                print_steps = "print",
                log_progress = False,
                plot_loss = False,
                num_workers = 10,
            )


### Load experiment data

In [ ]:
runs = (
    mlflow.search_runs(
        search_all_experiments = True,
        experiment_names = ["trainswt"],
        order_by = [f"params.F1_weighted_avg DESC"],
    )
    .assign( # Cast metrics to float numbers
        **{
            k: (lambda x, col=k: x[col].astype(np.float32))
            for k in [
                "params.F1_weighted_avg",
                "params.F1_micro_avg",
                "params.F1_macro_avg",
                "params.F1_samples_avg",
                "params.F1_snow",
                "params.F1_vegetation",
                "params.F1_anthropic",
                "params.F1_human",
                "params.F1_rock",
            ]
        }
    )
)

runs

### Select best backbone

In [ ]:
backbone_averages = (
    runs.groupby(["params.backbone"])
    .agg({"params.F1_weighted_avg": "mean"})
    .reset_index()
).sort_values("params.F1_weighted_avg", ascending=False)
backbone_averages

### Load best model

In [ ]:
best_run = (
    runs[runs["params.backbone"] == backbone_averages.iloc[0]["params.backbone"]]
    .sort_values("params.F1_weighted_avg", ascending=False)
    .iloc[0]
)

model = mlflow.pytorch.load_model(
    f"runs:/{best_run.run_id}/model",
    map_location=torch.device(ic.get_device()),
)
model.hr_desc()

### Display validation data for best model

In [ ]:
train, val = train_test_split(trainval,test_size=0.18,stratify=trainval["strat"],random_state=11)
train = train.drop("strat", axis=1)
val = val.drop("strat", axis=1)

model.get_val_data(dataset=ic.FldDataset(data=val, train_mode=False))["classification_report"]

### Final tests

In [ ]:
model.get_val_data(dataset=ic.FldDataset(data=test, train_mode=False))["classification_report"]

In [ ]:
model.get_val_data(dataset=ic.FldDataset(data=tab_challenge.reset_index(drop=True), train_mode=False))["classification_report"]

In [ ]:
model.get_val_data(dataset=ic.FldDataset(data=tab_challenge[tab_challenge['site'] == 'Carpathians'].reset_index(drop=True), 
                                         train_mode=False))["classification_report"]

In [ ]:
model.get_val_data(dataset=ic.FldDataset(data=tab_challenge[tab_challenge['site'] == 'French_Alps'].reset_index(drop=True), 
                                         train_mode=False))["classification_report"]

In [ ]:
model.get_val_data(dataset=ic.FldDataset(data=tab_challenge[tab_challenge['site'] == 'Stubai_Valley'].reset_index(drop=True), 
                                         train_mode=False))["classification_report"]

In [ ]:
model.get_val_data(dataset=ic.FldDataset(data=tab_challenge[tab_challenge['site'] == 'Vinschgau'].reset_index(drop=True), 
                                         train_mode=False))["classification_report"]

## End session

### Git

In [ ]:
subprocess.run(["cp", "Model_5_labels.ipynb", "AI/Model_5_labels.ipynb"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)

subprocess.run(["cp", "model.py", "AI/model.py"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)

subprocess.run(["cp", "install.sh", "AI/install.sh"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)

subprocess.run(["cp", "requirements.txt", "AI/requirements.txt"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)

### Bucket

In [ ]:
subprocess.run(["rm", "mlruns.zip"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)

subprocess.run(["zip", "-r", "mlruns.zip", "mlruns"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)

In [ ]:
s3.upload_file("mlruns.zip", mybucket, "mlruns.zip")
s3.upload_file("mlflow.db", mybucket, "mlflow.db")